# IST Scenario 1 — Did early aspirin improve outcome after acute ischaemic stroke?

_Reproducing the main result of the International Stroke Trial with the MIP Python client.
Level: first analysis. About 90 minutes._

**Learning goals.** Describe a trial population; compare an outcome between two randomised
groups with a cross-tabulation and a chi-square test; obtain an adjusted odds ratio from a
logistic regression; explain why randomisation makes adjustment almost unnecessary; state a
trial result in one careful sentence.

**Clinical background.** In the early 1990s it was not known whether aspirin in the first hours
after an ischaemic stroke helped or harmed. IST tested aspirin 300 mg daily for 14 days versus
no aspirin, started within 48 hours of onset. Together with the Chinese trial CAST, it is why
every patient with acute ischaemic stroke receives aspirin today, unless thrombolysis comes first.

**Before you start.** IST randomised 19,435 patients between 1991 and 1996. Nobody received
thrombolysis or thrombectomy, one third were randomised before a CT scan, and "dependent" was
measured by questionnaire, not by the modified Rankin Scale. The *conclusions* of the trial still
hold and are current practice; the *absolute percentages* describe 1990s stroke care and must not
be quoted to patients.

**How this notebook works.** Every analysis is a *federated* call: the code selects variables,
defines a cohort with filters and asks the MIP for aggregates (counts, means, a test, a model).
No patient row ever reaches this notebook. Run the cells in order; each step says what to look
for and what result to expect.

## 1. Connect to the MIP and open the IST pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("IST")      # add version="1.0" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())        # the pathology label differs: pick it from this list
    raise
ist = dm.datasets.list()[0]             # the IST pathology holds a single dataset
display(dm)
display(ist)

In [ ]:
# The variable tree follows the patient's path through the hospital: Demographics and event,
# Prestroke characteristics, Clinical assessment at admission, Admission imaging, Stroke
# characteristics, Acute treatment, Complications within 14 days, Hospitalization,
# Follow-up at 6 months, Death and survival, Predicted risk.
dm.tree(include_variables=True)

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the scenario; come back to it
# when you want to see how a result payload is unpacked.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    out = {}
    for item in raw:
        if isinstance(item, dict):
            out[str(item.get("code"))] = str(item.get("label"))
        else:
            out[str(item)] = str(item)
    return out


def analysis(variables, filters=None, drop_missing=False):
    """A Pipeline on the IST dataset with the given variables and cohort filter.
    Tests and models need drop_missing=True: the platform requires an explicit missing-values
    step and analyses complete cases only. describe() keeps it off so it can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=[ist], variables=variables), filters=filters,
                        handle_missing=handler)


def rows_for(result, variable):
    """The pooled 'describe' row of one variable (the platform adds one row per dataset as well)."""
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") == "all datasets"]


def numeric_summary(result, variables):
    """Mean, SD, median and range of numeric variables from a describe result."""
    rows = []
    for v in variables:
        for r in rows_for(result, v):
            d = r.get("data") or {}
            rows.append({"variable": v.label, "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "median": d.get("q2"),
                         "min": d.get("min"), "max": d.get("max")})
    return pd.DataFrame(rows)


def category_counts(result, variable):
    """Counts and percentages of the levels of a categorical variable from a describe result."""
    lab = labels_of(variable)
    rows = []
    for r in rows_for(result, variable):
        counts = (r.get("data") or {}).get("counts") or {}
        total = sum(counts.values())
        for c, n in counts.items():
            rows.append({"variable": variable.label, "level": lab.get(str(c), str(c)), "count": n,
                         "percent": 100 * n / total if total else np.nan})
    return pd.DataFrame(rows)


def describe_by(variables, group, base_filter=None):
    """Describe the variables separately in each level of `group` (one federated call per level).
    Returns (numeric table, categorical table) with a column per group level."""
    num, cat = [], []
    for c, label in labels_of(group).items():
        f = F(group) == c
        if base_filter is not None:
            f = f & base_filter
        res = analysis(variables, f).describe()
        n = numeric_summary(res, [v for v in variables if v.is_numerical()])
        if len(n):
            num.append(n.assign(group=label))
        for v in variables:
            if v.is_categorical():
                k = category_counts(res, v)
                if len(k):
                    cat.append(k.assign(group=label))
    num = pd.concat(num, ignore_index=True) if num else pd.DataFrame()
    cat = pd.concat(cat, ignore_index=True) if cat else pd.DataFrame()
    if len(num):
        num = num.pivot(index="variable", columns="group", values=["n", "mean", "sd"])
    if len(cat):
        cat = cat.pivot(index=["variable", "level"], columns="group", values="percent")
    return num, cat


def crosstab(exposure, outcome, base_filter=None, event="1"):
    """Cross-tabulation of outcome by exposure, as the MIP does it: counts come from one
    describe per exposure level, the p-value from the Chi-squared Test on the whole cohort."""
    lab_x, lab_y = labels_of(exposure), labels_of(outcome)
    rows = []
    for c, label in lab_x.items():
        f = (F(exposure) == c) & F(outcome).is_not_null()
        if base_filter is not None:
            f = f & base_filter
        res = analysis([exposure, outcome], f).describe()
        counts = {}
        for r in rows_for(res, outcome):
            counts = (r.get("data") or {}).get("counts") or {}
        n = sum(counts.values())
        if n == 0:
            continue
        row = {exposure.label: label, "n": n}
        for yc, ylabel in lab_y.items():
            row[ylabel] = counts.get(yc, 0)
        row["% " + lab_y.get(event, event)] = 100 * counts.get(event, 0) / n
        rows.append(row)
    table = pd.DataFrame(rows)
    f = F(exposure).is_not_null() & F(outcome).is_not_null()
    if base_filter is not None:
        f = f & base_filter
    chi = analysis([exposure, outcome], f, drop_missing=True).chi_square_test(x=exposure, y=outcome)
    stats = chi.raw if isinstance(chi.raw, dict) else {}
    table.attrs["chi2"] = stats.get("chi2")
    table.attrs["p"] = stats.get("p_value", stats.get("p"))
    table.attrs["outcome"] = outcome.label
    return table


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def show_crosstab(table, title=None):
    """Display a crosstab table, its chi-square test and a bar chart of the percentages."""
    pct_col = [c for c in table.columns if c.startswith("% ")][0]
    display(table.style.format({pct_col: "{:.1f}"}).hide(axis="index"))
    print(f"Chi-squared Test: chi2 = {table.attrs['chi2']:.2f}, p = {fmt_p(table.attrs['p'])}")
    ax = table.set_index(table.columns[0])[pct_col].plot(kind="bar", color="#4C72B0", figsize=(5, 3))
    ax.set_ylabel(pct_col)
    ax.set_title(title or f"{table.attrs['outcome']} by {table.columns[0]}")
    ax.set_xlabel("")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first level of the variable) that the platform leaves out."""
    by_code = {v._code: v for v in variables}

    def pretty(term):
        term = str(term)
        if "[" in term and term.endswith("]"):
            base, level = term[:-1].split("[", 1)
            v = by_code.get(base)
            return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
        v = by_code.get(term)
        return f"{v.label} (per unit)" if v else term

    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(pretty)
    for col in ("odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p": "{:.4f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


print("Toolbox ready.")

## 3. The variables of this scenario

| Role | Variable in the tree | Codes |
|---|---|---|
| exposure | Aspirin allocated (`RXASP`) | 0 no, 1 yes |
| primary outcome | Dead or dependent at 6 months (`DEADDEP6`, curation-derived) | 0 no, 1 yes |
| early outcome | Died within 14 days (`ID14`) | 0, 1 |
| benefit mechanism | Ischaemic stroke within 14 days (`ISC14`) | 0, 1 |
| harms | Cerebral bleed within 14 days (`H14`), Major non-cerebral bleed within 14 days (`TRAN14`) | 0, 1 |
| covariates | Age, Sex (1 male, 2 female), Conscious level (1 alert, 2 drowsy, 3 unconscious) | |

Variables are picked by their **label** in the tree. The code (for example `RXASP`) is what the
csv column is called; you will see it in some result payloads.

In [ ]:
rxasp  = dm.variables["Aspirin allocated"]
dd6    = dm.variables["Dead or dependent at 6 months"]
id14   = dm.variables["Died within 14 days"]
isc14  = dm.variables["Ischaemic stroke within 14 days"]
h14    = dm.variables["Cerebral bleed within 14 days"]
tran14 = dm.variables["Major non-cerebral bleed within 14 days"]
age    = dm.variables["Age"]
sex    = dm.variables["Sex"]
rconsc = dm.variables["Conscious level"]

display(mip.to_frame([rxasp, dd6, id14, isc14, h14, tran14, age, sex, rconsc]))

## Step 1. Describe the whole trial population

No filter: every randomised patient (intention to treat). `describe()` returns, for each
selected variable, the number of non-missing values and either mean/SD/quartiles (numeric) or
counts per level (categorical).

*Expected:* 19,435 patients; age 71.7 ± 11.6 years (median 73); 53.5% men; 21.9% drowsy and
1.3% unconscious; 50.0% allocated aspirin; 62.9% dead or dependent at six months among the
19,285 patients with a known six-month status.

In [ ]:
whole = analysis([age, sex, rconsc, rxasp, dd6]).describe()

display(numeric_summary(whole, [age]).style.format({"mean": "{:.1f}", "sd": "{:.1f}"}).hide(axis="index"))
for v in (sex, rconsc, rxasp, dd6):
    display(category_counts(whole, v).style.format({"percent": "{:.1f}"}).hide(axis="index"))

In [ ]:
# Age distribution, as a federated histogram (the MIP returns bins and counts only)
analysis([age], drop_missing=True).histogram(variable=age, bins=16).plot()
plt.show()

## Step 2. Check that randomisation worked

The same description, once in each allocation group. `describe_by()` runs one federated
`describe()` per level of the grouping variable and puts the results side by side.

*Expected:* the two groups are almost identical (age within 0.2 years, sex and consciousness
within one percentage point). Any later difference in outcome can be attributed to the
treatment, not to differences between the groups.

In [ ]:
num, cat = describe_by([age, sex, rconsc], group=rxasp)
display(num.style.format("{:.1f}"))
display(cat.style.format("{:.1f}"))

## Step 3. Primary outcome: aspirin versus dead or dependent at six months

`crosstab()` does what the MIP interface does when you cross two categorical variables: it
counts the outcome in each exposure level (a `describe()` per level) and runs the
*Chi-squared Test* on the whole cohort. Patients with an unknown six-month status are left
out by the `is_not_null()` filter, exactly as the platform drops them.

*Expected:* 6,000 of 9,639 (62.2%) with aspirin versus 6,125 of 9,646 (63.5%) without;
absolute difference 1.3 percentage points; chi-square 3.2, p = 0.07. This is the figure of the
1997 Lancet paper to the decimal.

In [ ]:
primary = crosstab(rxasp, dd6)
show_crosstab(primary, "Dead or dependent at 6 months, by aspirin allocation")

pct = primary.set_index("Aspirin allocated")["% Yes"]
print(f"Absolute difference: {pct['No'] - pct['Yes']:.1f} percentage points in favour of aspirin")

## Steps 4 to 6. Early death, mechanism of benefit, price of the benefit

The same cross-tabulation for four 14-day events. Read each as events per 1,000 patients.

*Expected:*

| Event within 14 days | Aspirin | No aspirin | p |
|---|---|---|---|
| Death | 872 (9.0%) | 909 (9.4%) | 0.35 |
| Recurrent ischaemic stroke | 153 (1.6%) | 242 (2.5%) | < 0.001 |
| Cerebral bleed | 87 (0.9%) | 74 (0.8%) | 0.31 |
| Major non-cerebral bleed | 109 (1.1%) | 57 (0.6%) | < 0.001 |

Aspirin prevented about 9 recurrent ischaemic strokes per 1,000 patients and caused about 5
extra major extracranial bleeds, most of them not fatal. That balance is why the effect on
death or dependency is small but positive.

In [ ]:
events = {"Death": id14, "Recurrent ischaemic stroke": isc14, "Cerebral bleed": h14, "Major non-cerebral bleed": tran14}
summary = []
for name, v in events.items():
    t = crosstab(rxasp, v)
    per1000 = t.set_index("Aspirin allocated")["% Yes"] * 10
    summary.append({"event within 14 days": name, "aspirin, per 1000": per1000["Yes"],
                    "no aspirin, per 1000": per1000["No"], "difference": per1000["Yes"] - per1000["No"],
                    "chi2": t.attrs["chi2"], "p": t.attrs["p"]})
summary = pd.DataFrame(summary)
display(summary.style.format({"aspirin, per 1000": "{:.0f}", "no aspirin, per 1000": "{:.0f}",
                              "difference": "{:+.0f}", "chi2": "{:.1f}", "p": fmt_p}).hide(axis="index"))

ax = summary.set_index("event within 14 days")[["aspirin, per 1000", "no aspirin, per 1000"]].plot(
    kind="barh", figsize=(7, 3.5), color=["#4C72B0", "#DD8452"])
ax.set_xlabel("events per 1,000 patients")
ax.set_ylabel("")
ax.set_title("14-day events by aspirin allocation")
plt.tight_layout()
plt.show()

## Step 7. Adjusted analysis with logistic regression

*Logistic Regression* with dependent variable `Dead or dependent at 6 months` (event = code 1)
and covariates aspirin, age, sex and conscious level. The platform treats nominal variables as
categorical and uses their **first level as reference** (no aspirin, male, fully alert), so each
odds ratio compares a level with that reference. Odds ratios and their 95% confidence
intervals are `exp(coefficient)` and `exp(coefficient ± 1.96 × SE)`.

*Expected (19,285 patients):* odds ratio for aspirin 0.94 (95% CI 0.88 to 1.00), p = 0.04.
Unadjusted (Step 3) the odds ratio is 0.95, p = 0.07. Each extra year of age multiplies the odds
of a poor outcome by 1.05; women have 1.4 times the odds of men; drowsy patients 5.7 times and
unconscious patients about 32 times the odds of alert patients.

Adjustment changed the aspirin estimate very little: in a randomised trial the covariates are
balanced (Step 2), so adjustment mainly sharpens precision. In observational data (Scenario 3)
adjustment is essential.

In [ ]:
covariates = [rxasp, age, sex, rconsc]
model = analysis([dd6] + covariates, F(dd6).is_not_null(), drop_missing=True).logistic_regression(
    x=covariates, y=dd6, positive_class="1")  # the event is code 1 = Yes

display(model)                                # the result card: N, features, hints
ors = odds_ratio_table(model, covariates)
show_odds_ratios(ors, "Odds of death or dependency at 6 months (adjusted)")

## Step 8. State the result

A model answer:

> In IST, allocation to aspirin 300 mg daily within 48 hours of stroke onset reduced death or
> dependency at six months from 63.5% to 62.2% (13 fewer per 1,000 treated, adjusted odds ratio
> 0.94, 95% CI 0.88 to 1.00), mainly by preventing early recurrent ischaemic stroke, at the cost
> of a small excess of major extracranial bleeding and no significant excess of intracranial
> haemorrhage. Because these patients received no reperfusion therapy and 1990s supportive care,
> the absolute outcome rates are not those expected today, but the conclusion that early aspirin
> is beneficial has been confirmed and is current practice.

**Questions for discussion**

1. The p-value is 0.07 in Step 3 and 0.04 in Step 7. Has the evidence changed? What matters
   more, the p-value or the size and consistency of the effect across outcomes?
2. Why did the trialists need 19,435 patients to detect a 1.3 percentage point difference? What
   would have happened with 2,000?
3. Some patients allocated to aspirin never took it (`Aspirin given for 14 days`), and some
   allocated to avoid it received it anyway. Why analyse by allocation (intention to treat) and
   not by treatment received?

## Optional extension: subgroups

Repeat Step 3 in patients with a CT-confirmed ischaemic stroke, then in the 599 patients whose
final diagnosis was a haemorrhagic stroke. A filter passed as `base_filter` restricts the cohort.
What does the second table say about the risk of giving aspirin before a scan, and about the
limits of a small subgroup?

In [ ]:
ctisch = dm.variables["CT-confirmed ischaemic stroke"]
fdiag = dm.variables["Final diagnosis of initial event"]

show_crosstab(crosstab(rxasp, dd6, base_filter=F(ctisch) == "1"), "CT-confirmed ischaemic stroke only")
show_crosstab(crosstab(rxasp, dd6, base_filter=F(fdiag) == "2"), "Final diagnosis haemorrhagic stroke only")